# Kokoro-82M — DIMER text-to-speech tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/kokoro-tts-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/kokoro-tts-pipeline/blob/main/tutorials/kokoro_tts_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-hexgrad%2FKokoro--82M-ffcc4d?style=flat)](https://huggingface.co/hexgrad/Kokoro-82M)
[![Upstream](https://img.shields.io/badge/Upstream-hexgrad%2Fkokoro-181717?style=flat&logo=github&logoColor=white)](https://github.com/hexgrad/kokoro)
[![arXiv](https://img.shields.io/badge/arXiv-2306.07691-b31b1b.svg)](https://arxiv.org/abs/2306.07691)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** text-to-speech (24 kHz mono float32 waveform from a named synthetic voice pack) using the pinned Kokoro-82M v1.0 weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API (`KokoroTTSPipeline`) rather than reimplementing model inference. At inference the `misaki` grapheme-to-phoneme library turns the text into a phoneme string, a 128-d style vector is read from the selected pre-computed voice pack (indexed by phoneme count), the StyleTTS 2 decoder predicts per-phoneme durations, pitch and energy, and the ISTFTNet vocoder renders one 24 kHz mono waveform per text segment; the segments are concatenated. **No adaptation occurs:** no training, fine-tuning, voice cloning, in-context conditioning, or preprocessing fitting — the only choice is which of the 54 shipped voice packs to use. What the upstream snapshot supplies is the checkpoint, the configuration and the voice packs; what this repository adds is manifest staging and SHA-256 verification of all 58 snapshot files, input validation and ceilings, one-language-per-instance voice checking, and a fixed output contract. **Speech quality has no intrinsic metric:** the pipeline ships no metric helper because naturalness (MOS) needs human listeners and intelligibility (ASR word error rate) needs an external recogniser; the notebook reports run-level facts (duration, peak amplitude, phonemes) and no quality score.

**Learning objectives:** bootstrap the repository in a fresh runtime, author a synthetic English sentence (or upload your own text), surface the pipeline's ceilings and voice/language rules, stage and digest-verify the immutable upstream snapshot including every voice pack, synthesise speech through the public API with an explicit seed, read the output contract correctly, write a playable WAV under `outputs/`, understand why no metric is reported and what external judges a real evaluation needs, and export machine-readable results plus provenance.

**This notebook does not demonstrate:** voice cloning or speaker adaptation (Kokoro has no speaker encoder and accepts no reference audio), speech-to-text (the `whisper-asr-pipeline` sibling covers that), voice mixing, SSML or emotion control, word-level timestamps, languages other than the pipeline's `lang_code` at load time, or any quality score. The repository exposes none of these.


## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (also float32; the pipeline does not change precision by device). The model card's CPU smoke loaded and verified the 58-file snapshot in 6.22 s and rendered 3.25 s of audio in 0.72 s, so the one-sentence default runs in seconds on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 355 MB snapshot (327 MB checkpoint plus 54 voice packs) are the largest downloads of the run.
- **Knowledge:** basic Python; what a phoneme string is; why a synthesised waveform has no ground truth to score against.
- **Data:** the default sample is one synthetic English sentence authored in code; BYOD is one UTF-8 text file, gated off by default. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API. The text you submit will be spoken verbatim.
- **External access:** the Git clone, the pinned wheel installs, the fetch of the git-ignored snapshot files from the Hugging Face Hub at the immutable revision, and — outside this package's control — the `misaki` G2P library's one-time download of the spaCy `en_core_web_sm` model on first English use. No credentials are needed.


## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`kokoro`, `misaki`, `torch`, `numpy`, `huggingface-hub`, `soundfile`) are pinned exactly by `pyproject.toml`. If installation replaces any package that this runtime has already imported (hosted runtimes commonly pre-import a different NumPy or Pillow), the cell fails with a restart instruction rather than continuing with mixed versions: restart the runtime and rerun from the top. Inference runs in float32 on both CPU and CUDA; no compilation or quantisation is applied. The `kokoro` library version is read from package metadata rather than imported here, because the notebook never calls the wrapped library directly. Look for a dictionary reporting the repository revision, Python, `torch`, `numpy`, `kokoro`, `misaki` and `soundfile` versions, and whether CUDA is available.


In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/kokoro-tts-pipeline.git'
REPO_NAME = 'kokoro-tts-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, numpy, torch
library_versions = {name: importlib.metadata.version(name) for name in ('kokoro', 'misaki', 'soundfile')}
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, **library_versions, 'cuda': torch.cuda.is_available()})

## 2. Author the sample text or optional BYOD

The default sample is **synthetic**: one English pangram written in this cell — the same sentence the model card's CPU smoke used, chosen because every word is in the `misaki` dictionary, so the espeak-ng fallback is not needed to pronounce it. It exists to prove the code path, not to measure anything: it ships **no reference recording**, so the audio it produces is smoke/sanity evidence that the pipeline works, never a quality measurement and never benchmark evidence. The voice, the speed and the seed are Colab form parameters so they can be changed without editing code; their allowed ranges are checked against the package in Section 3.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 text file of at most `MAX_TEXT_CHARS` characters; the pipeline splits it on newlines and synthesises each line as a segment, and the library truncates any single segment above 510 phonemes with a warning, so keep lines to a sentence or two. The upload stays inside this runtime. Numbers, acronyms and names outside the dictionary are pronounced by the espeak-ng fallback when it is available on the host and are otherwise dropped — Section 5 shows how to check.


In [ ]:
import hashlib
import io

USE_BYOD = False  # @param {type:"boolean"}
VOICE = 'af_heart'  # @param {type:"string"}
SPEED = 1.0  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    text = io.TextIOWrapper(io.BytesIO(uploaded[sample_name]), encoding='utf-8').read().strip()
    sample_kind = 'BYOD upload'
else:
    text = 'The quick brown fox jumps over the lazy dog.'
    sample_name = 'synthetic_pangram'
    sample_kind = 'synthetic (authored in this cell; the model card smoke sentence)'
text_sha256 = hashlib.sha256(text.encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'chars': len(text), 'lines': len(text.splitlines()), 'text_sha256': text_sha256, 'voice': VOICE, 'speed': SPEED, 'seed': SEED})
print(text[:200])

## 3. Validate the request against the pipeline ceilings

The pipeline enforces its operational limits at `synthesize()` time; this cell imports the same constants from the package so the values shown are the ones in force, and checks the request before any model work, naming the failing condition and the corrective action. `MAX_TEXT_CHARS` is the character ceiling per call; `MIN_SPEED`–`MAX_SPEED` is the allowed range of the duration multiplier; `LANG_CODES` are the nine language codes the library supports, of which `DEFAULT_LANG_CODE` (`a`, American English) is what this notebook loads — a voice must start with that letter (`af_…`/`am_…`), so a British `bf_…` voice is rejected by design rather than silently mixed; `SAMPLE_RATE` is the fixed 24 kHz output rate. Whether `VOICE` exists is checked in Section 4 against the verified manifest, which is the only authoritative voice list. The notebook does not trim or alter the text; the library's 510-phoneme segment cap is applied inside the pipeline and is only visible afterwards through the returned `phonemes`.


In [ ]:
from kokoro_tts_pipeline import DEFAULT_LANG_CODE, DEFAULT_VOICE, LANG_CODES, MAX_SPEED, MAX_TEXT_CHARS, MIN_SPEED, SAMPLE_RATE

ceilings = {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MIN_SPEED': MIN_SPEED, 'MAX_SPEED': MAX_SPEED, 'SAMPLE_RATE': SAMPLE_RATE, 'LANG_CODES': LANG_CODES, 'DEFAULT_LANG_CODE': DEFAULT_LANG_CODE, 'DEFAULT_VOICE': DEFAULT_VOICE}
print(ceilings)
problems = []
if not text.strip():
    problems.append('text is empty: supply at least one non-blank line')
if len(text) > MAX_TEXT_CHARS:
    problems.append(f'text has {len(text)} chars > MAX_TEXT_CHARS={MAX_TEXT_CHARS}: shorten or split the input')
if not MIN_SPEED <= SPEED <= MAX_SPEED:
    problems.append(f'SPEED={SPEED} is outside MIN_SPEED={MIN_SPEED}..MAX_SPEED={MAX_SPEED}: set a value in range')
if not VOICE or VOICE[0] != DEFAULT_LANG_CODE:
    problems.append(f"VOICE={VOICE!r} is not a lang_code={DEFAULT_LANG_CODE!r} voice: choose an 'af_'/'am_' voice for American English")
if problems:
    raise ValueError('input rejected before model execution: ' + '; '.join(problems))
print({'chars': len(text), 'speed': SPEED, 'voice': VOICE, 'lang_code': DEFAULT_LANG_CODE, 'within_ceilings': True})

## 4. Stage, verify and resolve the pinned model

Model acquisition goes through the package, not the notebook. The public API pins the exact upstream model repository and immutable 40-hex revision (`MODEL_ID`/`MODEL_REVISION` are imported from the package, never typed here). The Git repository commits the DIMER snapshot manifest (`weights/kokoro-82m/dimer-base-manifest.json`: model id, revision, and the byte size and SHA-256 of each of the 58 snapshot files), `config.json`, and the upstream `README.md`/`VOICES.md`, but git-ignores the 327 MB `kokoro-v1_0.pth` checkpoint and the 54 `voices/*.pt` packs, so a fresh clone must stage 55 files first. `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches only the manifest-listed files that are absent, from the Hub at the pinned revision, and returns the list it fetched (`[]` on a warm runtime); it refuses to stage if the committed manifest disagrees with the package's pinned identity. `verify_snapshot(WEIGHTS_DIR)` then re-hashes every listed file — checkpoint and all voice packs — and raises on the first size or digest mismatch; its returned dict is summarised, and `list_voices` reads the authoritative voice names from it.

**Trust boundary:** the checkpoint and the voice packs are PyTorch pickle files, not SafeTensors (`WEIGHT_FORMAT`). Digest verification runs before any file is opened, and the `kokoro` library then deserialises them with the weights-only loader (`LOADER_WEIGHTS_ONLY`), which restricts unpickling to tensors and primitive containers. Path-safety and digest checks do not make an unverified pickle safe; only the pinned, verified bytes ever reach the unpickler, and there is no fallback to a different download. `from_pretrained(weights_dir=WEIGHTS_DIR)` loads from that verified directory with `lang_code='a'` and reports `espeak_fallback`: whether the espeak-ng fallback for out-of-dictionary words bound on this host (the pinned `espeakng-loader` wheel bundles the library). The effective model identity, the voice count, the device and the fallback state are printed before inference. On first English use `misaki` may download the spaCy `en_core_web_sm` model — a network call made by the dependency, not by this package.


In [ ]:
from kokoro_tts_pipeline import LOADER_WEIGHTS_ONLY, MODEL_ID, MODEL_KEY, MODEL_REVISION, WEIGHT_FORMAT, KokoroTTSPipeline, list_voices, stage_missing_files, verify_snapshot

print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'weight_format': WEIGHT_FORMAT, 'loader_weights_only': LOADER_WEIGHTS_ONLY})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
# Only the manifest-listed files that are absent are fetched, at the immutable revision the
# package pins; verify_snapshot then checks every byte count and SHA-256 before anything is loaded.
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'fetched': len(fetched), 'first': fetched[:3], 'from': MODEL_ID, 'revision': MODEL_REVISION})
snapshot = verify_snapshot(WEIGHTS_DIR)
voices = list_voices(snapshot)
print({'snapshot_path': snapshot['path'], 'model_id': snapshot['modelId'], 'revision': snapshot['revision'], 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'voices': len(voices)})
if VOICE not in voices:
    raise ValueError(f'VOICE={VOICE!r} is not one of the {len(voices)} manifest voices: choose from {[v for v in voices if v[0] == DEFAULT_LANG_CODE]}')
pipe = KokoroTTSPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device, 'dtype': 'float32', 'lang_code': pipe.lang_code, 'source': pipe.source, 'espeak_fallback': pipe.espeak_fallback, 'voices': len(pipe.voices)})

## 5. Synthesise, write the WAV, and read the output correctly

`synthesize(text, voice=..., speed=...)` returns `audio` (a 1-D float32 NumPy array at `sample_rate` 24000 Hz), `num_samples`, `duration_s`, `peak_amplitude`, `segments` (one `graphemes`/`phonemes` pair per newline-split segment), the `voice`, `lang_code` and `speed` used, `espeak_fallback`, the device, and the model identity. **The output is not bit-deterministic by default:** the vocoder adds Gaussian noise and a random initial phase to its harmonic excitation, so two unseeded calls differ at the sample level (the model card measured a maximum absolute difference of 0.093 between consecutive smoke calls of identical length) while durations and phonemes stay the same; the pipeline sets no seed, so this cell calls `torch.manual_seed(SEED)` immediately before synthesis — the card recorded bit-identical waveforms across two seeded calls on the same host. Seeding does not make the audio identical across devices, PyTorch builds or CPU kernels.

**Evaluation:** the repository ships **no metric helper and reports no performance measure**. Speech quality has no ground truth to compare against: naturalness needs Mean Opinion Score ratings from human listeners, and intelligibility needs an external recogniser (for example re-transcribing the WAV with the `whisper-asr-pipeline` sibling and computing word error rate against the input text), both of which are judges this repository does not ship. The default sample has no reference recording, so **no metric is reported**; the sanity checks below (right dtype, rate and shape, finite samples, peak within full scale, one segment per line, no word dropped from the phoneme string) are falsifiable plumbing checks, not a quality result, and the model card's smoke observation (78,000 samples, 3.25 s, peak 0.342 on the same sentence, unseeded CPU) is quoted as one measurement on that host, not an expected value. The `phonemes` string is the only in-repository way to see whether the G2P dropped a word — with `espeak_fallback` `False`, out-of-dictionary words vanish silently. The WAV written to `outputs/` is 16-bit PCM at 24 kHz via the pinned `soundfile`; the inline player below appears only in an IPython front end.


In [ ]:
import time

import numpy as np
import soundfile as sf

torch.manual_seed(SEED)
started = time.perf_counter()
result = pipe.synthesize(text, voice=VOICE, speed=SPEED)
elapsed = time.perf_counter() - started
audio = result['audio']
checks = {
    'audio_is_float32_1d': isinstance(audio, np.ndarray) and audio.dtype == np.float32 and audio.ndim == 1,
    'sample_rate_matches_contract': result['sample_rate'] == SAMPLE_RATE,
    'all_samples_finite': bool(np.isfinite(audio).all()),
    'peak_within_full_scale': 0.0 < result['peak_amplitude'] <= 1.0,
    'one_segment_per_line': len(result['segments']) == len([line for line in text.splitlines() if line.strip()]),
    'duration_consistent': abs(result['duration_s'] - result['num_samples'] / SAMPLE_RATE) < 1e-6,
}
if not all(checks.values()):
    raise RuntimeError(f'synthesize output failed a sanity check: {checks}')
os.makedirs('outputs', exist_ok=True)
wav_path = 'outputs/kokoro_tts_sample.wav'
sf.write(wav_path, audio, result['sample_rate'], subtype='PCM_16')
wav_sha256 = hashlib.sha256(Path(wav_path).read_bytes()).hexdigest()
print({key: value for key, value in result.items() if key not in ('audio', 'segments')})
print({'seconds': round(elapsed, 3), 'audio_seconds_per_wall_second': round(result['duration_s'] / elapsed, 2), 'rms': round(float(np.sqrt(np.mean(np.square(audio)))), 4), 'checks': checks})
for index, segment in enumerate(result['segments']):
    print(f"segment {index}: graphemes={segment['graphemes'][:80]!r}")
    print(f"segment {index}: phonemes ={segment['phonemes'][:80]!r}")
print({'wav': wav_path, 'wav_sha256': wav_sha256, 'subtype': 'PCM_16'})
metrics = {}
print('no metric is reported: speech has no intrinsic ground truth, the sample has no reference recording, and the repository ships no metric helper; MOS needs human listeners and intelligibility needs an external ASR judge')
try:
    from IPython.display import Audio, display
    display(Audio(audio, rate=result['sample_rate']))
except ImportError:
    print('inline player unavailable outside an IPython front end; open the WAV file instead')

## 6. Export outputs and provenance

Two files are written under `outputs/`: the WAV from Section 5 and one JSON record with the WAV path and its SHA-256 (so the audio can be tied to this record), the run-level facts (`num_samples`, `duration_s`, `peak_amplitude`, RMS, wall time), the per-segment graphemes and phonemes, the request (`voice`, `lang_code`, `speed`, `seed`), `espeak_fallback`, the sanity checks, the ceilings in force, the empty metric block, the sample identity and text digest, the repository revision, the model identifier and immutable revision, the weight format and loader trust facts, the verified snapshot summary, and the runtime identity (Python, `torch`, `numpy`, `kokoro`, `misaki`, `soundfile`, device, dtype). No credentials are involved in any step, so none can reach the export.


In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
payload = {
    'wav': {'path': wav_path, 'sha256': wav_sha256, 'subtype': 'PCM_16', 'sample_rate': result['sample_rate']},
    'audio': {'num_samples': result['num_samples'], 'duration_s': result['duration_s'], 'peak_amplitude': result['peak_amplitude'], 'rms': float(np.sqrt(np.mean(np.square(audio))))},
    'segments': result['segments'],
    'request': {'voice': result['voice'], 'lang_code': result['lang_code'], 'speed': result['speed'], 'seed': SEED},
    'espeak_fallback': result['espeak_fallback'],
    'sanity_checks': checks,
    'ceilings': ceilings,
    'metrics': metrics,
    'sample': {'name': sample_name, 'kind': sample_kind, 'text': text, 'text_sha256': text_sha256},
    'seconds': round(elapsed, 3),
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'weight_format': WEIGHT_FORMAT,
    'loader_weights_only': LOADER_WEIGHTS_ONLY,
    'snapshot': {'path': snapshot['path'], 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'voices': len(voices)},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'numpy': numpy.__version__,
        **library_versions,
        'device': pipe.device,
        'dtype': 'float32',
    },
}
with open('outputs/kokoro_tts_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))
print('outputs/kokoro_tts_result.json')

## Interpretation and limits

The WAV is a synthetic rendering of the input text in one of 54 named synthetic voices: it is not a recording of any person, carries no quality score, and its only in-repository checks are structural (rate, dtype, finite samples, peak within full scale) plus the phoneme string that shows what the G2P actually voiced. On the synthetic sample the audio is plumbing evidence only; no metric is reported because none can be computed without an external judge, and a real evaluation needs MOS ratings from listeners or an ASR round-trip WER over a reference sentence set, with the judge named. The seed controls the vocoder noise on one host and build; it does not promise identical audio across devices. Out-of-dictionary words depend on the espeak-ng fallback and are dropped when it is absent; segments above 510 phonemes are truncated by the library; non-English `lang_code`s and non-English quality are not exercised here; the pipeline exposes no cloning, mixing, timestamps or emotion control. The checkpoint and voice packs are pickles loaded through the weights-only loader after digest verification — an operator who bypasses `verify_snapshot` and loads an unverified file takes on arbitrary-code-execution risk.

Successful execution proves that the recorded repository revision can bootstrap in a fresh runtime, stage and digest-verify the pinned model snapshot including all voice packs, validate the demonstrated request against the enforced ceilings, execute the public pipeline path, write a playable WAV, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, naturalness or intelligibility on any audience, safety for high-consequence read-outs, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 4: a staged file is incomplete or altered — delete it from `weights/kokoro-82m/` (or the affected `voices/*.pt`) and rerun Section 4. A `ValueError` naming `MAX_TEXT_CHARS`, `SPEED` or `VOICE` in Section 3 or 4: fix the form parameter or shorten the BYOD file and rerun from Section 2. `espeak_fallback: False` in Section 4 with words missing from the `phonemes` string in Section 5: the bundled espeak-ng library did not bind on this host — restrict the text to dictionary words or install `espeak-ng` system-wide and reload. A long pause in Section 4 on first use: `misaki` is downloading the spaCy `en_core_web_sm` model.

**Next experiments.** Change `VOICE` to another `af_`/`am_` pack (the printed voice list names them) and compare the renderings of the same sentence; set `SPEED` to 0.8 and 1.5 and compare `duration_s`; run the same seed twice and diff the two WAV digests to see the seeded determinism on your host; upload a paragraph with names and numbers via `USE_BYOD` and inspect the `phonemes` string for dropped words; re-transcribe the WAV with the `whisper-asr-pipeline` sibling and compute WER against the input as the first step towards an intelligibility number. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance: `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/hexgrad/Kokoro-82M
- Upstream code: https://github.com/hexgrad/kokoro (G2P: https://github.com/hexgrad/misaki)
- StyleTTS 2 (Li et al., 2023): https://arxiv.org/abs/2306.07691
- iSTFTNet (Kaneko et al., 2022): https://arxiv.org/abs/2203.02395
